# 📐 Statistical Inference — C1–C3

Notebook này dùng dữ liệu `commits_features.csv` để kiểm định các câu hỏi nghiên cứu. Kết quả gồm test statistic, p-value, effect size và khoảng tin cậy.

**Diễn giải:** mặc định dùng mức ý nghĩa α = 0,05. Nếu p-value < 0,05 thì bác bỏ giả thuyết không (có bằng chứng về sự khác biệt/liên hệ). Các kiểm định ở đây thực hiện trên từng dòng dữ liệu; vì một commit có thể xuất hiện ở nhiều dòng code/file, kết quả cần được diễn giải thận trọng về tính độc lập.

In [1]:
from pathlib import Path
import warnings
import numpy as np
import pandas as pd
from scipy import stats
warnings.filterwarnings('ignore')

DATA_PATH = Path('../data/processed/commits_features.csv')
df = pd.read_csv(DATA_PATH, low_memory=False)
df['commit_date'] = pd.to_datetime(df['commit_date'], utc=True, errors='coerce')
df['is_modified'] = df['is_modified'].astype(bool)
df['days_to_modification'] = pd.to_numeric(df['days_to_modification'], errors='coerce')
ALPHA = 0.05
REPORT_DIR = Path('../reports'); REPORT_DIR.mkdir(exist_ok=True)
print(f'Đã tải {len(df):,} dòng × {df.shape[1]} cột')

Đã tải 33,917 dòng × 24 cột


## C1 — Hypothesis testing

In [2]:
def cramers_v(table, chi2):
    n = table.to_numpy().sum(); r, k = table.shape
    return np.sqrt((chi2 / n) / max(1, min(k - 1, r - 1)))

def chi_square_result(name, row, col):
    table = pd.crosstab(df[row], df[col])
    chi2, p, dof, expected = stats.chi2_contingency(table)
    return {'question': name, 'test': 'Chi-square', 'statistic': chi2, 'p_value': p, 'dof': dof, 'effect_size': cramers_v(table, chi2), 'significant_α_0.05': p < ALPHA}

def mann_whitney_result(name, x, group):
    a = df.loc[(df[group] == False) & df[x].notna(), x]
    b = df.loc[(df[group] == True) & df[x].notna(), x]
    u, p = stats.mannwhitneyu(a, b, alternative='two-sided')
    # rank-biserial correlation: 0 = no separation, ±1 = complete separation
    rbc = 2 * u / (len(a) * len(b)) - 1
    return {'question': name, 'test': 'Mann–Whitney U', 'statistic': u, 'p_value': p, 'dof': np.nan, 'effect_size': rbc, 'n_group_0': len(a), 'n_group_1': len(b), 'median_group_0': a.median(), 'median_group_1': b.median(), 'significant_α_0.05': p < ALPHA}

c1_results = []
# C1.1, C1.2, C1.5, C1.6
c1_results += [chi_square_result('C1.1 language × is_modified', 'language', 'is_modified')]
c1_results += [chi_square_result('C1.2 year × is_modified', 'year', 'is_modified')]
# C1.3 cần so sánh theo language, không phải theo is_modified
days_py = df.loc[(df.language == 'python') & df.days_to_modification.notna(), 'days_to_modification']
days_js = df.loc[(df.language == 'js') & df.days_to_modification.notna(), 'days_to_modification']
u, p = stats.mannwhitneyu(days_py, days_js, alternative='two-sided')
c1_results += [{'question': 'C1.3 days_to_modification: Python vs JS', 'test': 'Mann–Whitney U', 'statistic': u, 'p_value': p, 'effect_size': 2*u/(len(days_py)*len(days_js))-1, 'n_python': len(days_py), 'n_js': len(days_js), 'median_python': days_py.median(), 'median_js': days_js.median(), 'significant_α_0.05': p < ALPHA}]
c1_results += [mann_whitney_result('C1.4 code_lines: is_modified False vs True', 'code_lines', 'is_modified')]
c1_results += [chi_square_result('C1.5 ai_tool × is_modified', 'ai_tool', 'is_modified')]
c1_results += [chi_square_result('C1.6 task_type × is_modified', 'task_type', 'is_modified')]
c1_table = pd.DataFrame(c1_results)
display(c1_table)
c1_table.to_csv(REPORT_DIR / 'statistical_c1_hypothesis_tests.csv', index=False)

,question,test,statistic,p_value,dof,effect_size,significant_α_0.05,n_python,n_js,median_python,median_js,n_group_0,n_group_1,median_group_0,median_group_1
0,C1.1 language × is_modified,Chi-square,3.590170e+01,2.075275e-09,1.0,0.032535,True,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,C1.2 year × is_modified,Chi-square,1.766150e+02,3.975844e-37,4.0,0.072161,True,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,C1.3 days_to_modification: Python vs JS,Mann–Whitney U,1.172944e+07,2.595299e-07,NaN,0.064727,True,7561.0,2914.0,1.766,0.8755,NaN,NaN,NaN,NaN
3,C1.4 code_lines: is_modified False vs True,Mann–Whitney U,1.107385e+08,1.634938e-160,NaN,-0.175153,True,NaN,NaN,NaN,NaN,21327.0,12590.0,28.0,43.0
4,C1.5 ai_tool × is_modified,Chi-square,3.802782e+02,4.031426e-78,7.0,0.105887,True,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
5,C1.6 task_type × is_modified,Chi-square,4.467762e+02,2.425305e-93,6.0,0.114772,True,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


## C2 — A/B testing, effect size và 95% confidence interval

Outcome dùng để so sánh là tỷ lệ `is_modified=True`. Cohen's h đo chênh lệch giữa hai tỷ lệ; CI dùng Wilson cho từng tỷ lệ.

In [3]:
def wilson_ci(success, n, confidence=.95):
    z = stats.norm.ppf(1 - (1-confidence)/2); phat = success/n; den = 1 + z*z/n
    center = (phat + z*z/(2*n))/den; half = z*np.sqrt(phat*(1-phat)/n + z*z/(4*n*n))/den
    return center-half, center+half

def cohens_h(p1, p2):
    return 2*np.arcsin(np.sqrt(p1)) - 2*np.arcsin(np.sqrt(p2))

def ab_test(name, mask_a, mask_b):
    a = df.loc[mask_a, 'is_modified']; b = df.loc[mask_b, 'is_modified']
    sa, sb = int(a.sum()), int(b.sum()); na, nb = len(a), len(b)
    table = np.array([[sa, na-sa], [sb, nb-sb]])
    chi2, p, _, _ = stats.chi2_contingency(table, correction=False)
    pa, pb = sa/na, sb/nb; lo_a, hi_a = wilson_ci(sa, na); lo_b, hi_b = wilson_ci(sb, nb)
    return {'comparison': name, 'group_a_n': na, 'group_b_n': nb, 'group_a_rate': pa, 'group_b_rate': pb, 'difference_a_minus_b': pa-pb, 'cohens_h': cohens_h(pa,pb), 'p_value': p, 'a_95CI': f'[{lo_a:.4f}, {hi_a:.4f}]', 'b_95CI': f'[{lo_b:.4f}, {hi_b:.4f}]', 'significant_α_0.05': p < ALPHA}

ab_results = [
    ab_test('C2.1 Before vs after ChatGPT', df.year.eq(2022), df.year.ge(2023)),
    ab_test('C2.2 Python vs JS', df.language.eq('python'), df.language.eq('js')),
    ab_test('C2.3 Weekend vs weekday', df.is_weekend.eq(True), df.is_weekend.eq(False)),
]
ab_table = pd.DataFrame(ab_results)
display(ab_table)
ab_table.to_csv(REPORT_DIR / 'statistical_c2_ab_tests.csv', index=False)

,comparison,group_a_n,group_b_n,group_a_rate,group_b_rate,difference_a_minus_b,cohens_h,p_value,a_95CI,b_95CI,significant_α_0.05
0,C2.1 Before vs after ChatGPT,106,33811,0.405660,0.371092,0.034568,0.070944,4.620287e-01,"[0.3171, 0.5008]","[0.3660, 0.3763]",False
1,C2.2 Python vs JS,23844,10073,0.381438,0.346967,0.034471,0.071653,1.923946e-09,"[0.3753, 0.3876]","[0.3377, 0.3563]",True
2,C2.3 Weekend vs weekday,7775,26142,0.348424,0.377974,-0.029550,-0.061456,2.192469e-06,"[0.3379, 0.3591]","[0.3721, 0.3839]",True


## C3 — Bootstrap confidence intervals

In [4]:
rng = np.random.default_rng(42)
def bootstrap_stat(values, statistic=np.median, n_boot=5000):
    values = np.asarray(values, dtype=float); estimates = np.empty(n_boot)
    for i in range(n_boot): estimates[i] = statistic(rng.choice(values, size=len(values), replace=True))
    return float(statistic(values)), np.quantile(estimates, .025), np.quantile(estimates, .975)

valid_days = df.loc[df.days_to_modification.notna() & (df.days_to_modification >= 0), 'days_to_modification'].to_numpy()
median, lo, hi = bootstrap_stat(valid_days, np.median)
bootstrap_median = pd.DataFrame([{'metric': 'median_days_to_modification', 'n': len(valid_days), 'estimate': median, 'ci_95_lower': lo, 'ci_95_upper': hi}])
print('C3.1 Bootstrap CI cho median days_to_modification:'); display(bootstrap_median)

year_rows = []
for year, group in df.groupby('year', dropna=True):
    values = group.is_modified.astype(float).to_numpy()
    estimate, lo, hi = bootstrap_stat(values, np.mean)
    year_rows.append({'year': int(year), 'n': len(values), 'modified_rate': estimate, 'ci_95_lower': lo, 'ci_95_upper': hi})
bootstrap_year = pd.DataFrame(year_rows).sort_values('year')
print('C3.2 Bootstrap CI cho tỷ lệ is_modified theo năm:'); display(bootstrap_year)
bootstrap_median.to_csv(REPORT_DIR / 'statistical_c3_median_bootstrap.csv', index=False)
bootstrap_year.to_csv(REPORT_DIR / 'statistical_c3_year_bootstrap.csv', index=False)

C3.1 Bootstrap CI cho median days_to_modification:


,metric,n,estimate,ci_95_lower,ci_95_upper
0,median_days_to_modification,10475,1.284,1.152,1.501


C3.2 Bootstrap CI cho tỷ lệ is_modified theo năm:


,year,n,modified_rate,ci_95_lower,ci_95_upper
0,2022,106,0.405660,0.311321,0.500000
1,2023,2579,0.443195,0.423420,0.461817
2,2024,6833,0.339529,0.328553,0.350651
3,2025,17388,0.353750,0.346618,0.360824
4,2026,7011,0.418343,0.406504,0.429753


## Cách viết kết luận

Với mỗi câu hỏi, báo cáo test, p-value, effect size và kết luận. Không viết 'chứng minh nguyên nhân'; chỉ viết 'có/không có bằng chứng thống kê về sự khác biệt/liên hệ'. Bootstrap CI mô tả độ bất định của ước lượng, không thay thế cho kiểm định nhân quả.